# state_conditioned_overlay_gap_to_20_v1

Purpose: diagnose why the highest-scored complete-WF candidate from `soxlbacktest_reference_model_score_v1` still fails the `<20%` walk-forward drawdown gate, and test simple position-throttle counterfactuals before spending time on another training run.

Governance:
- Canonical baseline: `soxl_backtest_policy_reimplementation_v1` / qlib `legacy_default` reference.
- Direct model source: `state_conditioned_drawdown_rebound_reward_train_v1`, selected by `soxlbacktest_reference_model_score_v1`.
- Risk floor reference: `wf_pass_ytd18_15_15_02` from `soxlbacktest_qlib_walkforward_baseline_v1`.
- Role: `diagnose`; this card does not promote an actor.
- Changed layer: execution/selection counterfactual only. No model weights or production behavior are changed.
- Acceptance check: identify whether a simple cap/risk-floor overlay can meet `worst WF MaxDD <20%`, `YTD >= SOXLBacktest`, and `YTD >= current-best alignment` at the same time.

Counterfactual caveat: overlay variants scale the model's reported daily NAV return by a position-throttle ratio. This is a path-level diagnostic, not a broker-accurate rerun. The original variant exactly reproduces the reported model NAV path by construction.


In [1]:

from __future__ import annotations

import json
import math
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "SPEC.md").exists():
            return candidate
    raise FileNotFoundError(f"Could not locate repo root from {start}")


ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CARD = ROOT / "runtime/model_selection/card/state_conditioned_overlay_gap_to_20_v1"
CARD.mkdir(parents=True, exist_ok=True)
PROFILE = "state_conditioned_drawdown_rebound_reward_train_v1"
MODEL_CARD = ROOT / "runtime/model/card" / PROFILE
BASELINE_CARD = ROOT / "runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1"
SCORE_CARD = ROOT / "runtime/model_selection/card/soxlbacktest_reference_model_score_v1"
ARTIFACTS = {
    "overlay_window_metrics": CARD / "overlay_window_metrics.csv",
    "overlay_gate_summary": CARD / "overlay_gate_summary.csv",
    "drawdown_intervals": CARD / "drawdown_intervals.csv",
    "variant_daily": CARD / "variant_daily.csv",
    "selection": CARD / "selection.json",
    "summary": CARD / "summary.json",
    "conclusion": CARD / "conclusion.md",
}


def rel(path: Path) -> str:
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)


print({"card": rel(CARD), "profile": PROFILE})


{'card': 'runtime/model_selection/card/state_conditioned_overlay_gap_to_20_v1', 'profile': 'state_conditioned_drawdown_rebound_reward_train_v1'}


In [2]:

HANDOFF = json.loads((ROOT / "runtime/model_selection/card/retraining_gate_gap_handoff_v1/training_objective_handoff.json").read_text())
SCORE_SUMMARY = json.loads((SCORE_CARD / "summary.json").read_text())
BASELINE_SUMMARY = json.loads((BASELINE_CARD / "summary.json").read_text())
REFERENCE_DAILY = pd.read_csv(BASELINE_CARD / "candidate_daily.csv")
REQUIRED_WF = list(HANDOFF["retention_gate"]["required_wf_windows"])
MAX_WF_DD = float(HANDOFF["retention_gate"]["max_walkforward_drawdown_pct"])
REQ_SOXL_RETURN = float(HANDOFF["required_soxlbacktest_return_pct"])
REQ_BEST_RETURN = float(HANDOFF["required_best_return_pct"])
YTD_START = SCORE_SUMMARY["thresholds"]["ytd_start"]
YTD_END = SCORE_SUMMARY["thresholds"]["ytd_end"]
thresholds = {
    "max_walkforward_drawdown_pct": MAX_WF_DD,
    "required_soxlbacktest_return_pct": REQ_SOXL_RETURN,
    "required_best_return_pct": REQ_BEST_RETURN,
    "required_wf_windows": REQUIRED_WF,
    "ytd_start": YTD_START,
    "ytd_end": YTD_END,
}
thresholds


{'max_walkforward_drawdown_pct': 20.0,
 'required_soxlbacktest_return_pct': 132.077978,
 'required_best_return_pct': 213.143462,
 'required_wf_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
 'ytd_start': '2026-01-02',
 'ytd_end': '2026-06-09'}

In [3]:

def normalise_daily(frame: pd.DataFrame) -> pd.DataFrame:
    df = frame.copy()
    if "date" not in df.columns and "datetime" in df.columns:
        df = df.rename(columns={"datetime": "date"})
    if "date" not in df.columns:
        raise ValueError("daily frame missing date/datetime column")
    if "close" not in df.columns and "price" in df.columns:
        df["close"] = df["price"]
    if "price" not in df.columns and "close" in df.columns:
        df["price"] = df["close"]
    if "action" not in df.columns:
        df["action"] = "HOLD"
    df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
    for col in ["strategy_nav", "price", "close", "position"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df["position"] = df.get("position", 0.0)
    df["action"] = df["action"].fillna("HOLD").astype(str).str.upper()
    return df.sort_values("date").reset_index(drop=True)


def max_drawdown_pct(nav: pd.Series | np.ndarray) -> float:
    values = np.asarray(nav, dtype=float)
    if len(values) == 0:
        return math.nan
    peaks = np.maximum.accumulate(values)
    return float(np.max((peaks - values) / peaks) * 100.0)


def return_pct(nav: pd.Series | np.ndarray) -> float:
    values = np.asarray(nav, dtype=float)
    if len(values) < 2 or values[0] == 0:
        return math.nan
    return float((values[-1] / values[0] - 1.0) * 100.0)


def drawdown_interval(daily: pd.DataFrame, nav_col: str = "strategy_nav", pos_col: str = "position") -> dict[str, Any]:
    df = daily.copy().reset_index(drop=True)
    nav = pd.to_numeric(df[nav_col], errors="coerce").to_numpy(dtype=float)
    if len(nav) == 0:
        return {}
    peaks = np.maximum.accumulate(nav)
    dd = (peaks - nav) / peaks
    trough_idx = int(np.nanargmax(dd))
    peak_idx = int(np.argmax(nav[: trough_idx + 1])) if trough_idx >= 0 else 0
    trough_nav = nav[trough_idx]
    recovery_idx = None
    for idx in range(trough_idx + 1, len(nav)):
        if nav[idx] >= peaks[trough_idx]:
            recovery_idx = idx
            break
    pos = pd.to_numeric(df[pos_col], errors="coerce").fillna(0.0)
    interval = df.iloc[peak_idx : trough_idx + 1]
    return {
        "peak_date": str(df.iloc[peak_idx]["date"]),
        "trough_date": str(df.iloc[trough_idx]["date"]),
        "recovery_date": "" if recovery_idx is None else str(df.iloc[recovery_idx]["date"]),
        "max_drawdown_pct": float(dd[trough_idx] * 100.0),
        "peak_nav": float(nav[peak_idx]),
        "trough_nav": float(trough_nav),
        "drawdown_days": int(trough_idx - peak_idx + 1),
        "avg_position_peak_to_trough": float(pos.iloc[peak_idx : trough_idx + 1].mean()) if len(interval) else math.nan,
        "max_position_peak_to_trough": float(pos.iloc[peak_idx : trough_idx + 1].max()) if len(interval) else math.nan,
    }


def load_model_window(window: str) -> pd.DataFrame:
    if window == "ytd_2026":
        path = MODEL_CARD / f"{PROFILE}_soxl_daily.csv"
        daily = normalise_daily(pd.read_csv(path))
        return daily[(daily["date"] >= YTD_START) & (daily["date"] <= YTD_END)].reset_index(drop=True)
    path = MODEL_CARD / f"{PROFILE}_{window}_soxl_daily.csv"
    return normalise_daily(pd.read_csv(path))


def load_risk_floor_window(window: str) -> pd.DataFrame:
    daily = normalise_daily(REFERENCE_DAILY[(REFERENCE_DAILY["candidate"].eq("wf_pass_ytd18_15_15_02")) & (REFERENCE_DAILY["window"].eq(window))])
    if window == "ytd_2026":
        daily = daily[(daily["date"] >= YTD_START) & (daily["date"] <= YTD_END)].reset_index(drop=True)
    return daily


windows = [*REQUIRED_WF, "ytd_2026"]
model_daily = {window: load_model_window(window) for window in windows}
risk_floor_daily = {window: load_risk_floor_window(window) for window in windows}
{window: (len(model_daily[window]), model_daily[window]["date"].min(), model_daily[window]["date"].max()) for window in windows}


{'wf_2020': (252, '2020-01-02', '2020-12-30'),
 'wf_2022': (250, '2022-01-03', '2022-12-29'),
 'wf_2023_2024': (251, '2024-01-02', '2024-12-30'),
 'ytd_2026': (109, '2026-01-02', '2026-06-09')}

In [4]:

def throttle_ratio(
    *,
    variant: str,
    model: pd.DataFrame,
    risk_floor: pd.DataFrame,
    current_nav: list[float] | None = None,
    daily_ret: np.ndarray | None = None,
    idx: int | None = None,
    params: dict[str, Any] | None = None,
) -> np.ndarray | float:
    params = params or {}
    pos = pd.to_numeric(model["position"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    if variant == "original_reported":
        return np.ones(len(model), dtype=float)
    if variant == "static_cap":
        cap = float(params["cap"])
        return np.where(pos > 1e-9, np.minimum(pos, cap) / pos, 0.0)
    if variant == "risk_floor_buffer":
        rf = risk_floor[["date", "position"]].rename(columns={"position": "risk_floor_position"})
        merged = model[["date", "position"]].merge(rf, on="date", how="left")
        floor_pos = pd.to_numeric(merged["risk_floor_position"], errors="coerce").ffill().fillna(0.0).to_numpy(dtype=float)
        cap = floor_pos + float(params["buffer"])
        return np.where(pos > 1e-9, np.minimum(pos, cap) / pos, 0.0)
    if variant == "drawdown_trigger_cap":
        assert current_nav is not None and idx is not None
        peak = max(current_nav) if current_nav else 0.0
        dd = ((peak - current_nav[-1]) / peak * 100.0) if peak else 0.0
        if dd >= float(params["threshold_dd_pct"]) and pos[idx] > 1e-9:
            return min(pos[idx], float(params["cap"])) / pos[idx]
        return 1.0
    raise ValueError(f"unknown variant: {variant}")


VARIANTS = [
    {"variant": "original_reported", "label": "original_reported", "params": {}},
    {"variant": "static_cap", "label": "static_cap_0_20", "params": {"cap": 0.20}},
    {"variant": "static_cap", "label": "static_cap_0_30", "params": {"cap": 0.30}},
    {"variant": "static_cap", "label": "static_cap_0_40", "params": {"cap": 0.40}},
    {"variant": "risk_floor_buffer", "label": "risk_floor_plus_0_00", "params": {"buffer": 0.00}},
    {"variant": "risk_floor_buffer", "label": "risk_floor_plus_0_10", "params": {"buffer": 0.10}},
    {"variant": "risk_floor_buffer", "label": "risk_floor_plus_0_20", "params": {"buffer": 0.20}},
    {"variant": "drawdown_trigger_cap", "label": "dd15_cap_0_20", "params": {"threshold_dd_pct": 15.0, "cap": 0.20}},
    {"variant": "drawdown_trigger_cap", "label": "dd15_cap_0_30", "params": {"threshold_dd_pct": 15.0, "cap": 0.30}},
    {"variant": "drawdown_trigger_cap", "label": "dd18_cap_0_30", "params": {"threshold_dd_pct": 18.0, "cap": 0.30}},
]


def simulate_variant(model: pd.DataFrame, risk_floor: pd.DataFrame, spec: dict[str, Any]) -> pd.DataFrame:
    df = model.copy().reset_index(drop=True)
    reported_nav = pd.to_numeric(df["strategy_nav"], errors="coerce").to_numpy(dtype=float)
    reported_ret = pd.Series(reported_nav).pct_change().fillna(0.0).to_numpy(dtype=float)
    nav = [float(reported_nav[0])]
    ratios = [1.0]
    if spec["variant"] in {"static_cap", "risk_floor_buffer", "original_reported"}:
        all_ratios = throttle_ratio(variant=spec["variant"], model=df, risk_floor=risk_floor, params=spec["params"])
        ratios = [float(all_ratios[0])]
        for idx in range(1, len(df)):
            ratio = float(all_ratios[idx])
            ratios.append(ratio)
            nav.append(nav[-1] * (1.0 + reported_ret[idx] * ratio))
    else:
        for idx in range(1, len(df)):
            ratio = float(throttle_ratio(
                variant=spec["variant"],
                model=df,
                risk_floor=risk_floor,
                current_nav=nav,
                daily_ret=reported_ret,
                idx=idx,
                params=spec["params"],
            ))
            ratios.append(ratio)
            nav.append(nav[-1] * (1.0 + reported_ret[idx] * ratio))
    out = df[["date", "price", "position", "strategy_nav", "action"]].copy()
    out["variant"] = spec["label"]
    out["overlay_nav"] = nav
    out["throttle_ratio"] = ratios
    out["overlay_position"] = pd.to_numeric(out["position"], errors="coerce").fillna(0.0).to_numpy(dtype=float) * np.asarray(ratios, dtype=float)
    out["reported_daily_return"] = reported_ret
    return out


window_rows = []
daily_frames = []
drawdown_rows = []
for spec in VARIANTS:
    for window in windows:
        sim = simulate_variant(model_daily[window], risk_floor_daily[window], spec)
        sim.insert(0, "window", window)
        daily_frames.append(sim)
        metrics = {
            "variant": spec["label"],
            "window": window,
            "return_pct": return_pct(sim["overlay_nav"]),
            "max_drawdown_pct": max_drawdown_pct(sim["overlay_nav"]),
            "avg_overlay_position": float(sim["overlay_position"].mean()),
            "max_overlay_position": float(sim["overlay_position"].max()),
            "avg_throttle_ratio": float(sim["throttle_ratio"].mean()),
            "rows": int(len(sim)),
        }
        window_rows.append(metrics)
        interval = drawdown_interval(sim.rename(columns={"overlay_nav": "nav", "overlay_position": "pos"}), nav_col="nav", pos_col="pos")
        interval.update({"variant": spec["label"], "window": window})
        drawdown_rows.append(interval)

overlay_window_metrics = pd.DataFrame(window_rows)
variant_daily = pd.concat(daily_frames, ignore_index=True)
drawdown_intervals = pd.DataFrame(drawdown_rows)
overlay_window_metrics.head(20)


/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_14008/4098286178.py:17: RuntimeWarning: invalid value encountered in divide
  return np.where(pos > 1e-9, np.minimum(pos, cap) / pos, 0.0)
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_14008/4098286178.py:17: RuntimeWarning: invalid value encountered in divide
  return np.where(pos > 1e-9, np.minimum(pos, cap) / pos, 0.0)
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_14008/4098286178.py:17: RuntimeWarning: invalid value encountered in divide
  return np.where(pos > 1e-9, np.minimum(pos, cap) / pos, 0.0)
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_14008/4098286178.py:17: RuntimeWarning: invalid value encountered in divide
  return np.where(pos > 1e-9, np.minimum(pos, cap) / pos, 0.0)
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_14008/4098286178.py:17: RuntimeWarning: invalid value encountered in divide
  return np.where(pos > 1e-9, np.minimum(pos, cap) / pos, 0.0)
/var/folde

,variant,window,return_pct,max_drawdown_pct,avg_overlay_position,max_overlay_position,avg_throttle_ratio,rows
0,original_reported,wf_2020,56.775553,24.413070,0.318823,0.865946,1.000000,252
1,original_reported,wf_2022,-23.275008,29.147985,0.186042,0.600000,1.000000,250
2,original_reported,wf_2023_2024,-3.160370,15.229639,0.110378,0.399777,1.000000,251
3,original_reported,ytd_2026,196.370488,31.390931,0.325608,0.909703,1.000000,109
4,static_cap_0_20,wf_2020,10.178581,12.499710,0.145764,0.200000,0.413977,252
5,static_cap_0_20,wf_2022,-17.851650,18.956740,0.132820,0.200000,0.610341,250
6,static_cap_0_20,wf_2023_2024,1.124356,10.257540,0.103503,0.200000,0.724454,251
7,static_cap_0_20,ytd_2026,8.696692,7.595172,0.089908,0.200000,0.136269,109
8,static_cap_0_30,wf_2020,14.784340,18.417006,0.208363,0.300000,0.557737,252
9,static_cap_0_30,wf_2022,-23.117036,23.868587,0.164562,0.300000,0.701841,250


In [5]:

gate_rows = []
for variant, group in overlay_window_metrics.groupby("variant", sort=False):
    wf = group[group["window"].isin(REQUIRED_WF)]
    ytd = group[group["window"].eq("ytd_2026")]
    worst_wf_dd = float(wf["max_drawdown_pct"].max())
    min_wf_return = float(wf["return_pct"].min())
    mean_wf_return = float(wf["return_pct"].mean())
    ytd_return = float(ytd.iloc[0]["return_pct"]) if not ytd.empty else math.nan
    ytd_dd = float(ytd.iloc[0]["max_drawdown_pct"]) if not ytd.empty else math.nan
    wf_dd_pass = bool(worst_wf_dd < MAX_WF_DD)
    soxl_return_pass = bool(ytd_return + 1e-6 >= REQ_SOXL_RETURN)
    best_return_pass = bool(ytd_return + 1e-6 >= REQ_BEST_RETURN)
    failures = []
    if not wf_dd_pass:
        failures.append("worst_wf_max_drawdown_ge_20")
    if not soxl_return_pass:
        failures.append("ytd_return_below_soxlbacktest_reference")
    if not best_return_pass:
        failures.append("ytd_return_below_current_best_alignment")
    gate_rows.append({
        "variant": variant,
        "worst_wf_max_drawdown_pct": round(worst_wf_dd, 6),
        "min_wf_return_pct": round(min_wf_return, 6),
        "mean_wf_return_pct": round(mean_wf_return, 6),
        "ytd_return_pct": round(ytd_return, 6),
        "ytd_max_drawdown_pct": round(ytd_dd, 6),
        "wf_drawdown_pass": wf_dd_pass,
        "ytd_soxlbacktest_return_pass": soxl_return_pass,
        "ytd_current_best_alignment_pass": best_return_pass,
        "retention_gate_pass": bool(wf_dd_pass and soxl_return_pass and best_return_pass),
        "failures": ";".join(failures),
    })

overlay_gate_summary = pd.DataFrame(gate_rows).sort_values(
    ["retention_gate_pass", "wf_drawdown_pass", "ytd_return_pct"],
    ascending=[False, False, False],
).reset_index(drop=True)
overlay_gate_summary


,variant,worst_wf_max_drawdown_pct,min_wf_return_pct,mean_wf_return_pct,ytd_return_pct,ytd_max_drawdown_pct,wf_drawdown_pass,ytd_soxlbacktest_return_pass,ytd_current_best_alignment_pass,retention_gate_pass,failures
0,static_cap_0_20,18.956740,-17.851650,-2.182904,8.696692,7.595172,True,False,False,False,ytd_return_below_soxlbacktest_reference;ytd_re...
1,risk_floor_plus_0_10,19.948353,-17.414445,-0.155303,4.401244,3.848353,True,False,False,False,ytd_return_below_soxlbacktest_reference;ytd_re...
2,risk_floor_plus_0_00,14.089214,-10.580423,-0.776293,0.000000,0.000000,True,False,False,False,ytd_return_below_soxlbacktest_reference;ytd_re...
3,original_reported,29.147985,-23.275008,10.113392,196.370488,31.390931,False,True,False,False,worst_wf_max_drawdown_ge_20;ytd_return_below_c...
4,dd18_cap_0_30,27.803341,-21.818904,8.271083,195.414088,31.390931,False,True,False,False,worst_wf_max_drawdown_ge_20;ytd_return_below_c...
5,dd15_cap_0_20,28.027275,-22.061401,10.247060,183.549207,31.390931,False,True,False,False,worst_wf_max_drawdown_ge_20;ytd_return_below_c...
6,dd15_cap_0_30,28.598784,-22.680283,9.496188,179.159504,31.390931,False,True,False,False,worst_wf_max_drawdown_ge_20;ytd_return_below_c...
7,static_cap_0_30,23.868587,-23.117036,-2.407970,12.535347,11.250990,False,False,False,False,worst_wf_max_drawdown_ge_20;ytd_return_below_s...
8,static_cap_0_40,26.722774,-25.999399,-2.275365,12.199978,14.830492,False,False,False,False,worst_wf_max_drawdown_ge_20;ytd_return_below_s...
9,risk_floor_plus_0_20,24.413070,-22.896394,-2.271591,8.696692,7.595172,False,False,False,False,worst_wf_max_drawdown_ge_20;ytd_return_below_s...


In [6]:

passing = overlay_gate_summary[overlay_gate_summary["retention_gate_pass"]]
wf_pass = overlay_gate_summary[overlay_gate_summary["wf_drawdown_pass"]]
ytd_pass = overlay_gate_summary[overlay_gate_summary["ytd_soxlbacktest_return_pass"]]
original = overlay_gate_summary[overlay_gate_summary["variant"].eq("original_reported")].iloc[0].to_dict()
best_wf_pass = wf_pass.sort_values("ytd_return_pct", ascending=False).iloc[0].to_dict() if not wf_pass.empty else {}
best_ytd_pass = ytd_pass.sort_values("worst_wf_max_drawdown_pct", ascending=True).iloc[0].to_dict() if not ytd_pass.empty else {}
selection = {
    "experiment": "state_conditioned_overlay_gap_to_20_v1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": "simple_position_throttles_do_not_satisfy_both_drawdown_and_return_gates",
    "profile": PROFILE,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "retention_gate_pass_variants": passing["variant"].tolist(),
    "wf_drawdown_pass_variants": wf_pass["variant"].tolist(),
    "ytd_soxlbacktest_return_pass_variants": ytd_pass["variant"].tolist(),
    "original_reported": original,
    "best_wf_drawdown_pass_variant_by_ytd": best_wf_pass,
    "best_ytd_pass_variant_by_wf_dd": best_ytd_pass,
    "thresholds": thresholds,
    "counterfactual_method": "scale_reported_model_daily_return_by_position_throttle_ratio; not broker-accurate promotion evidence",
    "sidecars": {name: rel(path) for name, path in ARTIFACTS.items()},
    "next_action": "do_not_use_simple_posthoc_cap_as_final_model; retrain or select with drawdown-aware objective/checkpoint gate that reduces wf_2020/wf_2022 exposure before drawdown rather than only after drawdown appears",
}
selection


{'experiment': 'state_conditioned_overlay_gap_to_20_v1',
 'created_at': '2026-06-13T11:11:16.896002+00:00',
 'decision': 'simple_position_throttles_do_not_satisfy_both_drawdown_and_return_gates',
 'profile': 'state_conditioned_drawdown_rebound_reward_train_v1',
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'retention_gate_pass_variants': [],
 'wf_drawdown_pass_variants': ['static_cap_0_20',
  'risk_floor_plus_0_10',
  'risk_floor_plus_0_00'],
 'ytd_soxlbacktest_return_pass_variants': ['original_reported',
  'dd18_cap_0_30',
  'dd15_cap_0_20',
  'dd15_cap_0_30'],
 'original_reported': {'variant': 'original_reported',
  'worst_wf_max_drawdown_pct': 29.147985,
  'min_wf_return_pct': -23.275008,
  'mean_wf_return_pct': 10.113392,
  'ytd_return_pct': 196.370488,
  'ytd_max_drawdown_pct': 31.390931,
  'wf_drawdown_pass': False,
  'ytd_soxlbacktest_return_pass': True,
  'ytd_current_best_alignment_pass': False,
  'retention_gate_pass': False,
  'failures': 'worst_wf_max_dra

In [7]:

overlay_window_metrics.to_csv(ARTIFACTS["overlay_window_metrics"], index=False)
overlay_gate_summary.to_csv(ARTIFACTS["overlay_gate_summary"], index=False)
drawdown_intervals.to_csv(ARTIFACTS["drawdown_intervals"], index=False)
variant_daily.to_csv(ARTIFACTS["variant_daily"], index=False)
ARTIFACTS["selection"].write_text(json.dumps(selection, indent=2, sort_keys=True) + "\n", encoding="utf-8")
summary = {
    "experiment": selection["experiment"],
    "created_at": selection["created_at"],
    "decision": selection["decision"],
    "profile": PROFILE,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "retention_gate_pass_variants": selection["retention_gate_pass_variants"],
    "wf_drawdown_pass_variants": selection["wf_drawdown_pass_variants"],
    "ytd_soxlbacktest_return_pass_variants": selection["ytd_soxlbacktest_return_pass_variants"],
    "original_reported": {
        "worst_wf_max_drawdown_pct": original["worst_wf_max_drawdown_pct"],
        "ytd_return_pct": original["ytd_return_pct"],
        "failures": original["failures"],
    },
    "best_wf_drawdown_pass_variant_by_ytd": best_wf_pass,
    "best_ytd_pass_variant_by_wf_dd": best_ytd_pass,
    "thresholds": thresholds,
    "sidecars": {name: rel(path) for name, path in ARTIFACTS.items()},
}
ARTIFACTS["summary"].write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
conclusion_lines = [
    "# state_conditioned_overlay_gap_to_20_v1",
    "",
    "- Decision: `simple_position_throttles_do_not_satisfy_both_drawdown_and_return_gates`.",
    f"- Original `{PROFILE}` path has worst WF MaxDD `{original['worst_wf_max_drawdown_pct']:.2f}%` and YTD return `{original['ytd_return_pct']:.2f}%`.",
    f"- WF-drawdown-pass variants: `{', '.join(selection['wf_drawdown_pass_variants']) if selection['wf_drawdown_pass_variants'] else 'none'}`.",
    f"- YTD SOXLBacktest-return-pass variants: `{', '.join(selection['ytd_soxlbacktest_return_pass_variants']) if selection['ytd_soxlbacktest_return_pass_variants'] else 'none'}`.",
    "- No tested simple cap, risk-floor throttle, or drawdown-trigger cap satisfies both WF drawdown and return gates.",
    "- Use this as branch evidence: a post-hoc simple cap is not enough; next work should change training/selection so exposure is lower before the bad WF drawdown intervals while preserving current YTD upside.",
    "",
    "## Sidecars",
]
conclusion_lines.extend(f"- `{rel(path)}`" for path in ARTIFACTS.values())
ARTIFACTS["conclusion"].write_text("\n".join(conclusion_lines) + "\n", encoding="utf-8")
summary


{'experiment': 'state_conditioned_overlay_gap_to_20_v1',
 'created_at': '2026-06-13T11:11:16.896002+00:00',
 'decision': 'simple_position_throttles_do_not_satisfy_both_drawdown_and_return_gates',
 'profile': 'state_conditioned_drawdown_rebound_reward_train_v1',
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'retention_gate_pass_variants': [],
 'wf_drawdown_pass_variants': ['static_cap_0_20',
  'risk_floor_plus_0_10',
  'risk_floor_plus_0_00'],
 'ytd_soxlbacktest_return_pass_variants': ['original_reported',
  'dd18_cap_0_30',
  'dd15_cap_0_20',
  'dd15_cap_0_30'],
 'original_reported': {'worst_wf_max_drawdown_pct': 29.147985,
  'ytd_return_pct': 196.370488,
  'failures': 'worst_wf_max_drawdown_ge_20;ytd_return_below_current_best_alignment'},
 'best_wf_drawdown_pass_variant_by_ytd': {'variant': 'static_cap_0_20',
  'worst_wf_max_drawdown_pct': 18.95674,
  'min_wf_return_pct': -17.85165,
  'mean_wf_return_pct': -2.182904,
  'ytd_return_pct': 8.696692,
  'ytd_max_drawdown

## Conclusion

This diagnostic keeps promotion blocked. It shows that pure post-hoc throttles split into two bad families: low cap/risk-floor variants can pass the walk-forward drawdown gate only by destroying YTD return, while drawdown-triggered caps preserve much of the YTD return but do not reduce the worst walk-forward drawdown enough. The next model loop needs drawdown-aware training/selection that acts before the bad intervals, not a simple after-the-fact execution cap.
